In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4','cuda:7'])

('cuda:1', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78094)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 58183)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 51669)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 30149)

# #### Device() ####
# device = cuda:1



In [2]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:1)
# edge_attr                (32464, 16)              Tensor (cuda:1)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:1)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path              PosixPath
# metadata_path            PosixPath
# gene_name_path           PosixPath
# metadata_complete        (1231, 93)               DataFrame
# metadata                 (1172, 2)                DataFrame
# y                        (1172,)                  Tensor (cuda:1)
# y_labels                 5                        list
# ensgv                    (60660, 3)             

In [3]:
from modules.layers import MultiheadSetPooling
from modules.loss import MultiLoss, KLDLoss
from modules.model import BaseAutoencoder
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer

from functools import partial
from torch_geometric.nn import GCNConv, GATConv
import torch.nn as nn

In [ ]:
## Model

model_template = partial(
    BaseAutoencoder,
    dataset = dataset,
    embed_dim = 128,
    num_heads=4,
    hidden_dims = 1,
    method='node',
    mlp=False,
    variational=True,

    # layers
    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True},
)

model_grid = grid(
    model_template,
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    lr=1e-3,
    trainer_norm_class=LogCounts,
    out_keys={'input':'x_t_pred', 'target':'x_t', 'z_mu':'z_mu', 'z_logvar':'z_logvar'},
    loss_class=MultiLoss,
    loss_kwargs={
        'loss_classes':[nn.MSELoss, KLDLoss],
        'loss_inputs':[
            {'input':'input', 'target':'target'}, 
            {'mu':'z_mu', 'logvar':'z_logvar'}
        ],
        'ema_norm':False
    },
    early_stop=True,
    stop_metric='mae',
)

trainer_grid = grid(
    trainer_template,
    merge_keys = 'loss_kwargs',
    loss_kwargs=[
        {'loss_weights': (1,1e-5)},
        {'loss_weights': (1,7.5e-6)},
        {'loss_weights': (1,5e-6)},
        {'loss_weights': (1,2.5e-6)},
        {'loss_weights': (1,1e-6)},
    ]
)

## Experiment
expt = Experiment(
    num_trials=10,
    num_epochs=300,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

5


In [20]:
expt.run_experiment(
    'reconstr_grid4c2',
    report_metrics=['loss','kld','rmse','mae','r2'],
    # save_csv=True,
    # save_params=True,
    # save_values=True,
    # verbose=False,
)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/300 [00:00<?, ?it/s]

Trial 1/10, Config: 1/5 (losskwargsLossweights11e-5),	 loss=0.4631    kld=184.5126    rmse=0.7433    mae=0.4938    r2=0.8546


  0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 